# 8.8. Designing Convolution Network Architectures
D2L의 Designing Convolution Network Architectures장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 지금까지 CNN은 어떻게 발전했는가?

지금까지 살펴본 CNN들은 대부분 연구자의 직관과 실험을 통해서 발전했다.

`AlexNet` → CNN을 깊게 만들면 성능이 좋아질 수 있음  
`VGG` → 작은 3×3 Conv를 반복해서 쌓기  
`NiN` → 1×1 Conv 활용 → Global Average Pooling  
`GoogLeNet` → 여러 크기의 Conv를 병렬로 사용  
`ResNet` → Skip Connection으로 매우 깊은 모델 학습  
`ResNeXt` → Grouped Convolution 

CNN 설계에는 반복적으로 나타나는 몇 가지 좋은 패턴이 존재한다. 그러면 좋은 CNN 구조를 매번 사람의 감으로 설계해야 할까?

이것을 체계적으로 탐색하려는 방법 중 하나가 Neural Architecture Search(NAS)​이다.

## 2. Neural Architecture Search

NAS는 여러 신경망 구조를 자동으로 탐색해서 좋은 구조를 찾는 방법이다.

예를 들어서 다음과 같은 것들을 자동으로 바꿔본다.

```text
Conv 몇 개?
채널 몇 개?
Block 몇 개?
Kernel 크기는?
Skip Connection은?
```

문제는 탐색해야 할 조합이 너무 많다.

    모델 생성 -> 학습 -> 성능 측정 -> 다른 모델 생성 -> 다시 학습 -> ...

계산 비용이 너무 크다. 하지만 EfficientNet 같은 모델은 NAS를 이용해 얻어진 대표적인 결과 중 하나이다.

이번 장에서는 단 하나의 최고의 모델을 찾는 대신 좋은 모델들이 공통으로 가지는 설계 규칙을 찾는다. 이 과정에서 만들어진 모델이 RegNet이다.

## 3. 하나의 모델보다 Design Space

기존 NAS의 목표를 단순하게 보면

    수많은 모델 -> 가장 좋은 모델 하나 찾기

RegNet은 접근이 다르다.

    수많은 모델 -> 좋은 모델이 가지는 공통 특성 찾기 -> 좋은 모델이 많이 나오는 설계 공간 만들기

이것을 Network Design Space라고 한다. 

최고의 CNN은 무엇인가? 보단 좋은 CNN들은 어떤 구조적 규칙을 갖고 있는가?

## 4. AnyNet의 기본 구조

먼저 다양한 CNN을 표현할 수 있는 기본 틀을 만든다. 이것을 AnyNet이라고 한다. 구조는 크게 세 부분이다.

```text
Input -> Stem -> Body -> Head -> Output
```

### Stem

이미지를 처음 받아 간단한 특징을 추출하고 해상도를 줄인다.

3×3 Conv stride=2 -> BatchNorm -> ReLU

### body

실제 특징 추출의 대부분을 담당한다. 여러개의 Stage로 구성된다.

```text
Body
├─ Stage 1
├─ Stage 2
├─ Stage 3
└─ Stage 4
```
각 Stage 안에는 여러 개의 ResNeXt Block이 들어간다.

### Head

마지막 특징을 실제 분류 결과로 변환한다.

```text
Global Average Pooling
↓
Flatten
↓
Linear
↓
클래스 예측
```

## 5. Stage란 무엇인가?

CNN을 설계할 때 Conv를 무작정 일렬로 나열하기보다 비슷한 크기의 feature map을 처리하는 부분을 하나의 Stage로 묶는 경우가 많다.

예를 들어 ImageNet의 224×224 이미지라면

```text
입력
224×224
↓ Stem
112×112
↓ Stage 1
56×56
↓ Stage 2
28×28
↓ Stage 3
14×14
↓ Stage 4
7×7
```

Stage가 바뀔 때마다 H, W를 절반으로 줄이는 구조다.

각 Stage의 첫 번째 Block에서 stride = 2를 사용한다.

이후 같은 Stage 안의 나머지 Block들은 stride = 1을 사용하여 H, W를 유지한다.

Stage 시작 -> 해상도 감소  
Stage 내부 -> 같은 해상도에서 특징 추출

## 6. CNN을 결정하는 주요 Hyperparameter

AnyNet에선 각 Stage마다 여러 값을 정할 수 있다.

### Depth

$$
d_i
$$

해당 Stage에 Block을 몇 개 넣을 것인가?

d₁ = 2 -> Stage 1에 Block 2개

### Channel

$$
c_i
$$

해당 Stage에서 몇 개의 채널을 사용할 것인가?

c₁ = 64 -> Stage 1 출력 채널 64

### Groups

$$
g_i
$$

ResNeXt의 Grouped Convolution을 몇 개의 그룹으로 나눌 것인가?

### Bottleneck Ratio

$$
k_i
$$

Block 내부에서 채널을 얼마나 압축할 것인가? CNN 구조 하나를 만드는 것은 결국 이런 값들을 정하는 문제라고 볼 수 있다.

```text
Stage 1 → depth, channels, groups, bottleneck
Stage 2 → depth, channels, groups, bottleneck
Stage 3 → depth, channels, groups, bottleneck
Stage 4 → depth, channels, groups, bottleneck
```

## 7. 문제는 조합이 너무 많다.

AnyNet에는 총 17개 정도의 설계 파라미터가 존재한다. 각 파라미터마다 두 개의 선택지만 있다고 가정해도 2^17 = 131072개의 모델이 나온다.

그리고 각 모델을 실제로 학습해 비교해야 한다면 비용이 엄청나게 커진다.

그래서 모든 조합 탐색이 아니라 `성능을 떨어뜨리지 않으면서 불필요한 자유도를 제거하는 방향`으로 Design Space를 줄여나간다.

## 8. 모델을 끝까지 학습하지 않아도 된다.

모든 후보 모델을 완전히 학습시키는 것도 낭비가 크다.

예를 들어서

```text
예를 들어 원래:

모델 A → 100 epoch
모델 B → 100 epoch
모델 C → 100 epoch
...
```

해야한다면 비용이 너무 크다. 대신 초기 몇 epoch의 성능을 이용한다.

```text
모델 A → 조금 학습 → 괜찮음
모델 B → 조금 학습 → 나쁨
모델 C → 조금 학습 → 괜찮음
```

좋지 않은 후보를 빠르게 제거할 수 있다. 적은 자원으로 얻은 중간 결과를 이용하여 최종 성능을 추정하는 방법을 Multi-Fidelity Optimization이라고 한다.

작은 모델에서 발견한 좋은 설계 규칙이 큰 모델에서도 어느 정도 유지된다고 가정한다. 그래서 작은 모델로 먼저 실험하여 탐색 비용을 줄인다.

## 9. 좋은 Design Space 비교하기

여기서 중요한 것은 특정 모델 하나의 정확도가 아니다.

예를 들어서 이렇다고 했을때

```text
Design A에서 모델 100개 생성
Design B에서 모델 100개 생성
```

Design A -> 대부분 좋은 성능  
Design B -> 몇 개만 좋고 대부분 나쁜 성능

이라면 Design A가 더 좋은 설계 공간이라고 볼 수 있다. 이것을 수학적으로 평가하기 위해 오류의 누적분포함수(CDF)를 사용한다.

$$
F(e,p)=P_{\text{net}\sim p}{e(\text{net})\le e}
$$

> 이 Design Space에서 뽑은 모델이 특정 error 이하일 확률은 얼마나 되는가?

를 나타낸다.

실제로 모든 모델을 조사할 수 없으므로 여러 모델을 표본으로 뽑아 empirical CDF를 사용한다.

$$
\hat{F}(e)=\frac{1}{n}\sum_{i=1}^{n}\mathbf{1}(e_i\le e)
$$

좋은 모델 하나 찾기아니라 좋은 모델이 자주 나오는 설계 규칙 찾는 것이다.

## 10. 실험으로 발견한 CNN 설계 규칙

여러 AnyNet 구조를 실험하면서 중요한 규칙들이 발견되었다.

첫 번째는 모든 Stage에서 Bottleneck Ratio를 다르게 할 필요가 없다는 것이다.

$$
k_1=k_2=k_3=k_4=k
$$

두 번째로 Group Width 역시 Stage마다 다르게 할 필요가 없었다.

$$
g_1=g_2=g_3=g_4=g
$$

이렇게 하면 여러 Hyperparameter를 하나로 줄일 수 있다.

더 중요한 결과는 네트워크가 깊어질수록 채널 수를 증가시키는 것이 좋았다는 것이다.

$$
c_i \le c_{i+1}
$$

```text
Stage 1: 32 channels
Stage 2: 64 channels
Stage 3: 128 channels
Stage 4: 256 channels
```

뒤로 갈수록 채널을 늘리는 구조다.

그리고 Stage가 진행될수록 Block의 수를 증가시키는 것도 좋은 성능을 보였다.

$$
d_i \le d_{i+1}
$$

```text
Stage 1: Block 2개
Stage 2: Block 3개
Stage 3: Block 5개
Stage 4: Block 7개
```

## 11. RegNet의 핵심 설계 규칙

앞의 실험을 통해 Design Space를 계속 좁히며 RegNet이 만들어졌다.

핵심 규칙은 다음과 같다.

1. Stage별 Bottleneck Ratio를 통일한다.
2. Stage별 Group Width를 통일한다.
3. 깊어질수록 채널 수를 증가시킨다.
4. 깊어질수록 Stage의 Depth를 증가시킨다.

특히 채널 수는 Block이 깊어질수록 대략 선형적으로 증가하는 것이 좋다는 결과를 얻었다.

$$
c_j \approx c_0+c_a j
$$

$c_0$: 초기 채널 수  
$c_a$: 채널 증가량  
$j$: Block 위치  

얕은 Layer -> 적은 채널  
깊은 Layer -> 많은 채널

우리가 지금까지 CNN에서 자주 본 구조가 단순한 관습 뿐 아니라 실험적으로도 좋은 설계라는 뜻이다.

또 Bottleneck Ratio는 굳이 Bottleneck으로 채널을 압축하지 않는 것이 좋았다.

## 12. RegNetX 예제

D2L에서는 간단한 RegNetX 모델을 다음과 같은 설정으로 사용한다.

```text
Stem channels = 32

Stage 1
depth = 4
channels = 32

Stage 2
depth = 6
channels = 80

groups = 16
bottleneck ratio = 1

구조를 단순하게 보면:

입력
[1, 1, 96, 96]

↓ Stem
[1, 32, 48, 48]

↓ Stage 1
[1, 32, 24, 24]

↓ Stage 2
[1, 80, 12, 12]

↓ Global Average Pooling + Linear
[1, 10]
```

깊어질수록 H, W는 낮아지고 Channel은 높아진다. 이미지를 공간적으로 압축하며 더 많은 종류의 특징을 표현하도록 만든다.

## 13. CNN 설계에서 정말 중요한 것

CNN을 설계할 때 완전히 무작위로

```text
Conv 3개 넣어볼까?
Pool 넣어볼까?
채널 128로 할까?
```

하는 것이 아니라 어느 정도 검증된 설계 원칙이 존재한다.

```text
입력
↓
초기 특징 추출

Stage 1
낮은 수준 특징
높은 H,W
적은 Channel

↓

Stage 2

↓

Stage 3

↓

Stage 4
복잡한 특징
낮은 H,W
많은 Channel

↓
Global Average Pooling
↓
Classifier
```

이런 같은 구조가 반복적으로 나타난다.

CNN 설계는 제작자 마음대로 할 수는 있지만 아무렇게나 만드는 것은 아니다. 좋은 성능을 보였던 구조적 패턴과 계산량을 고려하면서 Depth, Channel, Resolution 등을 결정한다.

## 14. CNN과 Vision Transformer

CNN에는 이미지 처리에 적합한 강한 가정이 들어가 있다.

대표적인 것은 이거다.

- Locality: 가까운 픽셀끼리 중요한 관계가 있을 것이다.
- Translation Invariance: 물체 위치가 조금 이동해도 같은 특징으로 볼 수 있어야 한다.

그래서 CNN은 오랫동안 컴퓨터 비전의 핵심 구조였다. 하지만 대규모 데이터와 계산 자원을 사용할 수 있게 되면서 Vision Transformer가 매우 강력한 성능을 보이기 시작했다.

Transformer는 CNN보다 이미지에 대한 사전 가정이 적다.

    CNN -> 이미지에 적합한 구조를 사람이 미리 넣어줌  
    Vision Transformer -> 상대적으로 적은 사전 가정 -> 데이터로 관계를 더 많이 학습

따라서 데이터와 계산량이 충분히 크다면 모델이 직접 구조를 학습하는 방식이 강력할 수 있다는 것이 현대 비전 모델 발전의 중요한 흐름이다.

## 15. 오늘의 정리

- CNN 아키텍처는 단순히 Conv를 마음대로 쌓는 것이 아니다.
- 현대 CNN은 Stem → Body → Head 구조로 생각할 수 있다.
- Body는 여러 Stage로 나누어 구성하는 경우가 많다.
- Stage가 바뀔 때 보통 H, W를 줄인다.
- 네트워크가 깊어질수록 채널 수를 증가시키는 것이 일반적으로 유리하다.
- Stage별 Block 수 역시 뒤쪽으로 갈수록 증가시킬 수 있다.
- AnyNet은 다양한 CNN 구조를 표현하기 위한 넓은 Design Space다.
- 모든 구조를 직접 탐색하면 조합 수가 너무 많아 계산 비용이 크다.
- RegNet은 최고의 모델 하나를 찾기보다 좋은 모델들이 공유하는 설계 규칙을 찾는다.
- Bottleneck Ratio와 Group Width는 Stage마다 다르게 정할 필요가 없었다.
- RegNet 실험에서는 깊어질수록 Channel ↑이라는 설계가 효과적이었다.
- 작은 모델과 짧은 학습을 이용하면 CNN 구조 탐색 비용을 크게 줄일 수 있다.
- 결국 CNN 설계는 Depth, Channel, Resolution, Block 구조, 계산량 사이의 균형을 잡는 문제다.
- 이후 Vision Transformer에서는 CNN보다 훨씬 약한 이미지 사전 가정을 사용하면서 데이터로 관계를 학습하는 방향으로 발전한다.